In [2]:
import struct
import sys
from transformers import AutoModelForCausalLM

src = "/home/bgi-chao/repl/Qwen2.5-1.5B-Instruct"
dst = "model.bin"

model = AutoModelForCausalLM.from_pretrained(
    src,
    torch_dtype="auto"
)
c = model.config
dim = c.hidden_size
hidden_dim = c.intermediate_size
n_layers = c.num_hidden_layers
n_heads = c.num_attention_heads
n_kv_heads = c.num_key_value_heads
vocab_size = c.vocab_size
sd = model.state_dict()

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/bgi-chao/repl/Qwen2.5-1.5B-Instruct'. Use `repo_type` argument if needed.

In [ ]:
# 也可以按字段逐项查看
for name, value in c.to_dict().items():
    print(f"{name}: {value}")

transformers_version: 5.16.1
architectures: ['Qwen2ForCausalLM']
output_hidden_states: False
return_dict: True
dtype: bfloat16
chunk_size_feed_forward: 0
is_encoder_decoder: False
id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1}
problem_type: None
vocab_size: 151936
hidden_size: 1536
intermediate_size: 8960
num_hidden_layers: 28
num_attention_heads: 12
num_key_value_heads: 2
hidden_act: silu
max_position_embeddings: 32768
initializer_range: 0.02
rms_norm_eps: 1e-06
use_cache: True
tie_word_embeddings: True
rope_parameters: {'rope_theta': 1000000.0, 'rope_type': 'default'}
use_sliding_window: False
sliding_window: None
max_window_layers: 21
layer_types: ['full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'fu

In [5]:
for name , param in sd.items():
    print(name, param.shape, param.dtype)

model.embed_tokens.weight torch.Size([151936, 1536]) torch.bfloat16
model.layers.0.self_attn.q_proj.weight torch.Size([1536, 1536]) torch.bfloat16
model.layers.0.self_attn.q_proj.bias torch.Size([1536]) torch.bfloat16
model.layers.0.self_attn.k_proj.weight torch.Size([256, 1536]) torch.bfloat16
model.layers.0.self_attn.k_proj.bias torch.Size([256]) torch.bfloat16
model.layers.0.self_attn.v_proj.weight torch.Size([256, 1536]) torch.bfloat16
model.layers.0.self_attn.v_proj.bias torch.Size([256]) torch.bfloat16
model.layers.0.self_attn.o_proj.weight torch.Size([1536, 1536]) torch.bfloat16
model.layers.0.mlp.gate_proj.weight torch.Size([8960, 1536]) torch.bfloat16
model.layers.0.mlp.up_proj.weight torch.Size([8960, 1536]) torch.bfloat16
model.layers.0.mlp.down_proj.weight torch.Size([1536, 8960]) torch.bfloat16
model.layers.0.input_layernorm.weight torch.Size([1536]) torch.bfloat16
model.layers.0.post_attention_layernorm.weight torch.Size([1536]) torch.bfloat16
model.layers.1.self_attn.q_p

In [ ]:
import torch
from transformers import AutoTokenizer

del sd
model = model.float().eval() # set model to eval mode, and convert the parameters to float32
sd = model.state_dict()

In [ ]:
# tokenizer 
tokenizer = AutoTokenizer.from_pretrained(src)

# Instruct 模型：先套聊天模板，再编码。
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "hello"}],
    tokenize=False,
    add_generation_prompt=True,
)

# tokenizer full picture: text(the user prompt) -> token(Qwen use BPE algorithms to split text into tokens, each token has an uniq token_id) -> ids(the tensor type of token_ids) -> X(the embedding of token_ids)
print(text)
token_ids = tokenizer.encode(text, add_special_tokens=False)
print(token_ids)
print(len(token_ids))
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(tokens)
ids = torch.tensor(
    token_ids,
    dtype=torch.long,
    device=sd["model.embed_tokens.weight"].device,
)
print(ids.shape)
X = sd["model.embed_tokens.weight"][ids] # vocabulary embedding lookup: find the 30 token in 151936 token-vocabulary
print(X.shape)


In [ ]:
w = model.model.layers[0].state_dict()
for weight, param in w.items():
    print(weight, param.shape, param.dtype)

# X的shape: torch.Size([30, 1536])，表示有30个token，每个token的embedding维度是1536
head_dim = dim // n_heads
x = X

'''
forward begin
'''
# 1. RMSNorm：每个 token 的向量独立归一化
x1 = x / torch.sqrt((x * x).mean(-1, keepdim=True) + c.rms_norm_eps)
x1 = x1 * w["input_layernorm.weight"]

In [ ]:
print("tensor shape after RMSNorm:", x1.shape)

In [ ]:
# show the Q K V weights shape:
print("Q weight shape:", w["self_attn.q_proj.weight"].shape)
print("K weight shape:", w["self_attn.k_proj.weight"].shape)
print("V weight shape:", w["self_attn.v_proj.weight"].shape)


# 2. Self-Attention
# 2.1 Q/K/V 投影：权重按 [输出维度, 输入维度] 存储，因此转置
q = x1 @ w["self_attn.q_proj.weight"].T + w["self_attn.q_proj.bias"]# w is a dictionary containing the weights of the first layer of model
k = x1 @ w["self_attn.k_proj.weight"].T + w["self_attn.k_proj.bias"]
v = x1 @ w["self_attn.v_proj.weight"].T + w["self_attn.v_proj.bias"]
print("q/k/v tensor(after QKV matrix mul):", q.shape, k.shape, v.shape)

# 拆分注意力头, 128在这里代表把token的1536embedding拆分成12个不同的语义空间
q = q.reshape(X.shape[0], n_heads, head_dim).transpose(0, 1)     # [12, N, 128]
k = k.reshape(X.shape[0], n_kv_heads, head_dim).transpose(0, 1)  # [ 2, N, 128]
v = v.reshape(X.shape[0], n_kv_heads, head_dim).transpose(0, 1)  # [ 2, N, 128]
print("after split the head, q's shape:", q.shape)
print("after split the head, k's shape:", k.shape)
print("after split the head, v's shape:", v.shape)

# 2.2 RoPE：根据位置旋转 Q、K；本模型 theta = 1,000,000
position = torch.arange(X.shape[0], device=x.device).float()
index = torch.arange(0, head_dim, 2, device=x.device).float()
frequency = 1.0 / (1_000_000.0 ** (index / head_dim))
angle = position[:, None] * frequency[None, :]         # [N, 64]
angle = torch.cat([angle, angle], dim=-1)              # [N, 128]

def rope(t):
    first, second = t.chunk(2, dim=-1)
    rotated = torch.cat([-second, first], dim=-1)
    return t * angle.cos() + rotated * angle.sin()

q, k = rope(q), rope(k)
print("q, k after RoPE do not change:", q.shape, k.shape)

# GQA：每6个Q头共享一个K/V头，当做Q和K/V的计算时再复制到和Q-head相同的个数
k = k.repeat_interleave(n_heads // n_kv_heads, dim=0)  # [12, N, 128]
v = v.repeat_interleave(n_heads // n_kv_heads, dim=0)  # [12, N, 128]
print("k, v after GQA:", k.shape, v.shape) 

# 2.3 QKᵀ / √d：每个位置与所有位置计算匹配分数
scores = (q @ k.transpose(-2, -1)) / (head_dim ** 0.5)  # [12, N, N]

# 2.4 causal mask + softmax：屏蔽未来位置，再归一化
mask = torch.ones(X.shape[0], X.shape[0], dtype=torch.bool, device=x.device).triu(1)
scores = scores.masked_fill(mask, float("-inf"))
attention = torch.softmax(scores, dim=-1)
print("   attention:", attention.shape)

# 2.5 对 V 加权求和，再拼接各头
a = attention @ v                                    # [12, N, 128]
a = a.transpose(0, 1).reshape(X.shape[0], dim)                  # [N, 1536]

# 2.6 输出投影
a = a @ w["self_attn.o_proj.weight"].T
print("2. Attention:", a.shape)

In [ ]:
# 3. 第一次 Residual
x = x + a
print("3. Residual:", x.shape)

In [ ]:
# 4. 第二个 RMSNorm
x2 = x / torch.sqrt((x * x).mean(-1, keepdim=True) + c.rms_norm_eps)
x2 = x2 * w["post_attention_layernorm.weight"]
print("4. RMSNorm:", x2.shape)

In [ ]:
# 5. MLP / SwiGLU
g = x2 @ w["mlp.gate_proj.weight"].T                   # [N, 8960]
u = x2 @ w["mlp.up_proj.weight"].T                     # [N, 8960]
h = (g * torch.sigmoid(g)) * u                        # SiLU(g) * u
y = h @ w["mlp.down_proj.weight"].T                   # [N, 1536]
print("5. MLP:", g.shape, "->", y.shape)

In [ ]:
# 6. 第二次 Residual：Y 将作为下一层的输入
Y = x + y
print("6. Layer output:", Y.shape)